In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import CanineModel, CanineTokenizer
import pandas as pd
from classes.single_encoder import SingleEncoder
from classes.conversational_dataset import ConversationDataset
from torch.utils.data import DataLoader
from tqdm import tqdm
from collections import defaultdict
from torch.utils.data import Sampler
import random
import os

device = "cuda" if torch.cuda.is_available() else "cpu"

Skipping import of cpp extensions due to incompatible torch version 2.7.1+cu118 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0114 14:59:54.072000 45312 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
# Author Contrastive Loss (ACL) implementation
def author_contrastive_loss(z, authors, temperature=0.1):
    """
    z: [B, D] normalized embeddings
    authors: [B] author ids
    """
    z = F.normalize(z, dim=1)
    sim = z @ z.T / temperature  # [B, B]

    # mask self similarity
    mask = torch.eye(sim.size(0), device=sim.device).bool()
    sim = sim.masked_fill(mask, -1e9)

    # positives: same author
    pos_mask = authors.unsqueeze(0) == authors.unsqueeze(1)
    pos_mask = pos_mask & ~mask  # remove self

    # log-softmax over rows
    log_prob = F.log_softmax(sim, dim=1)

    # average over positives
    loss = -log_prob[pos_mask].mean()
    return loss

In [3]:
# Balanced Batch Sampler based on authors

def build_author_index_from_rows(rows, author2id):
    author_to_indices = defaultdict(list)
    for idx, row in enumerate(rows):
        author_id = author2id[row["Author"]]
        author_to_indices[author_id].append(idx)
    return author_to_indices


class AuthorBalancedBatchSampler(Sampler):
    def __init__(
        self,
        dataset,
        authors_per_batch=8,
        samples_per_author=4
    ):
        self.dataset = dataset
        self.authors_per_batch = authors_per_batch
        self.samples_per_author = samples_per_author

        self.author_to_indices = build_author_index_from_rows(
            dataset.rows,
            dataset.author2id
        )

        self.authors = list(self.author_to_indices.keys())
        self.batch_size = authors_per_batch * samples_per_author

    def __iter__(self):
        random.shuffle(self.authors)

        for i in range(0, len(self.authors), self.authors_per_batch):
            batch_authors = self.authors[i:i + self.authors_per_batch]
            batch_indices = []

            for author in batch_authors:
                indices = self.author_to_indices[author]
                if len(indices) >= self.samples_per_author:
                    chosen = random.sample(indices, self.samples_per_author)
                else:
                    chosen = random.choices(indices, k=self.samples_per_author)
                batch_indices.extend(chosen)

            yield batch_indices

    def __len__(self):
        return len(self.authors) // self.authors_per_batch
    

def collate_fn(batch):
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "author": torch.tensor([b["author"] for b in batch]),
        "content": [b["content"] for b in batch]
    }

In [4]:
rows = pd.read_csv("../data/train/retriever_train_undersampled.csv")
rows['Content'] = rows['Content'].astype("string")
rows = rows.to_dict(orient="records")

tokenizer = CanineTokenizer.from_pretrained("google/canine-s")
dataset = ConversationDataset(rows, tokenizer)

sampler = AuthorBalancedBatchSampler(
    dataset=dataset,
    authors_per_batch=8,
    samples_per_author=4
)


loader = DataLoader(
    dataset,
    batch_sampler=sampler,
    collate_fn=collate_fn,
)


In [5]:
from collections import Counter

def preview_batches(loader, num_batches=3):
    for i, batch in enumerate(loader):
        authors = batch["author"].tolist()
        counts = Counter(authors)

        print(f"\nBatch {i+1}")
        print("Batch size:", len(authors))
        print("Unique authors:", len(counts))
        print("Samples per author:", dict(counts))

        if i + 1 >= num_batches:
            break

preview_batches(loader, num_batches=2)


Batch 1
Batch size: 32
Unique authors: 8
Samples per author: {87: 4, 209: 4, 127: 4, 86: 4, 217: 4, 23: 4, 215: 4, 52: 4}

Batch 2
Batch size: 32
Unique authors: 8
Samples per author: {50: 4, 218: 4, 169: 4, 213: 4, 111: 4, 34: 4, 233: 4, 98: 4}


In [6]:
model = SingleEncoder().to(device)

optimizer = torch.optim.AdamW([
    {"params": model.encoder.parameters(), "lr": 1e-5},
    {"params": model.proj.parameters(), "lr": 5e-4}
])

epochs = 5
temperature = 0.1

model.train()

for epoch in range(epochs):
    total_loss = 0.0

    for batch in tqdm(loader, desc=f"Epoch {epoch+1}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        authors = batch["author"].to(device)

        z = model(input_ids, attention_mask)

        loss = author_contrastive_loss(
            z=z,
            authors=authors,
            temperature=temperature
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f}")

save_dir = "../output/models/author_contrastive"

model.eval()
model.cpu() 

Epoch 1: 31it [00:15,  1.99it/s]                        


Epoch 1 | Loss: 3.5718


Epoch 2: 31it [00:14,  2.10it/s]                        


Epoch 2 | Loss: 3.4576


Epoch 3: 31it [00:15,  2.06it/s]                        


...

Epoch 4: 31it [00:15,  2.01it/s]                        


Epoch 4 | Loss: 3.4218


Epoch 5: 31it [00:14,  2.16it/s]                        


Epoch 5 | Loss: 3.4150


SingleEncoder(
  (encoder): CanineModel(
    (char_embeddings): CanineEmbeddings(
      (HashBucketCodepointEmbedder_0): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_1): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_2): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_3): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_4): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_5): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_6): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_7): Embedding(16384, 96)
      (char_position_embeddings): Embedding(16384, 768)
      (token_type_embeddings): Embedding(16, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (initial_char_encoder): CanineEncoder(
      (layer): ModuleList(
        (0): CanineLayer(
          (attention): CanineAttention(
            (self): CanineSelfAttention(
              (query): Linear

In [8]:
# Save model properly
save_dir = "../output/models/author_contrastive"
os.makedirs(save_dir, exist_ok=True)

# Save the complete model configuration
torch.save({
    'model_state_dict': model.state_dict(),
    'proj_dim': model.proj[-1].out_features,
    'model_type': 'SingleEncoder',
    'model_name': 'google/canine-s'  # Add base model name
}, os.path.join(save_dir, "single_encoder.pt"))

# Save encoder and tokenizer
model.encoder.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

# Save projection head separately (optional)
torch.save(model.proj.state_dict(), os.path.join(save_dir, "projection_head.pt"))

print(f"Model saved to {save_dir}")

Model saved to ../output/models/author_contrastive
